In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", header=None, names=["label", "text"])

print(df.head())

  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [ ]:
# 1. How many rows and columns?
print(df.shape)

# 2. How many spam and ham?
print(df["label"].value_counts())

# 3. See some spam examples
print(df[df["label"] == "spam"].head(5))

(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64
   label                                               text
2   spam  Free entry in 2 a wkly comp to win FA Cup fina...
5   spam  FreeMsg Hey there darling it's been 3 week's n...
8   spam  WINNER!! As a valued network customer you have...
9   spam  Had your mobile 11 months or more? U R entitle...
11  spam  SIX chances to win CASH! From 100 to 20,000 po...


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

small = [
    "win free prize",
    "free free entry",
    "meeting tomorrow"
    "have a free gift"
    "free hamper"

]

cv = CountVectorizer()
numbers = cv.fit_transform(small)

print(cv.get_feature_names_out())
print(numbers.toarray())

['entry' 'free' 'giftfree' 'hamper' 'meeting' 'prize' 'tomorrowhave' 'win']
[[0 1 0 0 0 1 0 1]
 [1 2 0 0 0 0 0 0]
 [0 1 1 1 1 0 1 0]]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

Training messages: 4457
Testing messages: 1115


In [ ]:
print(y_train.value_counts())

label
ham     3859
spam     598
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

print("Practice messages:", len(X_train))
print("Exam messages:", len(X_test))

Practice messages: 4457
Exam messages: 1115


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Turn words into numbers
vectorizer = TfidfVectorizer(stop_words="english")
X_train_nums = vectorizer.fit_transform(X_train)
X_test_nums = vectorizer.transform(X_test)

# Teach the model
model = MultinomialNB()
model.fit(X_train_nums, y_train)

print("Training finished!")

Training finished!


In [ ]:
from sklearn.metrics import accuracy_score

guesses = model.predict(X_test_nums)

print("Accuracy:", accuracy_score(y_test, guesses))

Accuracy: 0.97847533632287


In [ ]:
my_message = ["Congratulations! You won a free gift card. Click now!"]

my_numbers = vectorizer.transform(my_message)
print(model.predict(my_numbers))

['spam']


In [ ]:
my_message = ["you have a meeting assigned tommorow at 10 am!"]

my_numbers = vectorizer.transform(my_message)
print(model.predict(my_numbers))

['ham']


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, guesses))
print(classification_report(y_test, guesses))

[[966   0]
 [ 24 125]]
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       1.00      0.84      0.91       149

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import confusion_matrix, classification_report

svm = LinearSVC()
svm.fit(X_train_nums, y_train)

svm_guesses = svm.predict(X_test_nums)

print(confusion_matrix(y_test, svm_guesses))
print(classification_report(y_test, svm_guesses))

[[965   1]
 [ 10 139]]
              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       966
        spam       0.99      0.93      0.96       149

    accuracy                           0.99      1115
   macro avg       0.99      0.97      0.98      1115
weighted avg       0.99      0.99      0.99      1115



In [ ]:
import numpy as np

words = vectorizer.get_feature_names_out()
scores = svm.coef_[0]

order = np.argsort(scores)

print("TOP 15 SPAM WORDS:")
for i in order[-15:][::-1]:
    print(words[i])

print()
print("TOP 15 NORMAL (HAM) WORDS:")
for i in order[:15]:
    print(words[i])

TOP 15 SPAM WORDS:
uk
www
claim
txt
50
150p
service
mobile
ringtone
stop
dating
reply
tones
88066
ac

TOP 15 NORMAL (HAM) WORDS:
fullonsms
liked
lt
gt
road
mail
later
ll
way
said
morning
ok
home
sorry
bus


In [ ]:
def check(message):
    numbers = vectorizer.transform([message])
    answer = svm.predict(numbers)[0]
    print(answer.upper(), "<-", message)

check("Congratulations! You won a free gift card. Click now!")
check("Hi, are we meeting tomorrow at 5?")
check("URGENT! Claim your prize, call 09061234567 now")
check("Can you send me the notes from class?")

SPAM <- Congratulations! You won a free gift card. Click now!
HAM <- Hi, are we meeting tomorrow at 5?
SPAM <- URGENT! Claim your prize, call 09061234567 now
HAM <- Can you send me the notes from class?


In [ ]:
check("Hey, is the free coffee still on at the office?")
check("Your bank account is locked. Verify your details at the link")
check("Mom, please call me when you are free")

HAM <- Hey, is the free coffee still on at the office?
HAM <- Your bank account is locked. Verify your details at the link
HAM <- Mom, please call me when you are free


In [ ]:
import joblib

joblib.dump(svm, "spam_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("Saved!")

Saved!


In [1]:
!pip install gradio -q

import gradio as gr

def check_spam(message):
    numbers = vectorizer.transform([message])
    answer = svm.predict(numbers)[0]
    if answer == "spam":
        return "🚨 SPAM"
    return "✅ Normal message (ham)"

app = gr.Interface(
    fn=check_spam,
    inputs=gr.Textbox(lines=6, label="Paste your email or message here"),
    outputs=gr.Textbox(label="Verdict"),
    title="Spam Email Checker",
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76ef7f10c4828b3e8a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

extra_spam = [
    "ur winning 1000rs prize money claim now",
    "You have won Rs 50000 lottery! Call now to claim your prize",
    "Congratulations! You won 5000rs cash prize. Click the link",
    "Get free recharge of Rs 500 now. Click here to claim",
    "Your KYC is pending. Update now or account will be blocked",
    "Win Rs 10000 cash. Reply YES to claim your reward",
    "Urgent! Your bank account is blocked. Verify details at link",
    "Free gift worth Rs 2000 waiting for you. Claim today",
    "you are luckydraw from companyxyz and you won 20lakh INR"
]
extra_ham = [
    "Ok I will call you after dinner",
    "Can you send me 500 rs for the auto fare",
    "Meeting is at 5, please bring the documents",
    "I reached home, will message you tomorrow",
]

extra = pd.DataFrame({
    "label": ["spam"] * len(extra_spam) + ["ham"] * len(extra_ham),
    "text": extra_spam + extra_ham,
})

# repeat the new examples so the model pays attention to them
df2 = pd.concat([df] + [extra] * 20, ignore_index=True)

X_train, X_test, y_train, y_test = train_test_split(
    df2["text"], df2["label"], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(stop_words="english")
X_train_nums = vectorizer.fit_transform(X_train)

svm = LinearSVC(class_weight="balanced")
svm.fit(X_train_nums, y_train)

print("Retrained!")

Retrained!


In [ ]:
check("ur winning 1000rs prize money")
check("Can you send me 500 rs for lunch")

SPAM <- ur winning 1000rs prize money
HAM <- Can you send me 500 rs for lunch


In [ ]:
import joblib

joblib.dump(svm, "spam_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("Saved!")

Saved!
